In [1]:
!pip install --upgrade langchain-openai
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install --upgrade langchain_huggingface


!pip install qdrant-client rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 841.6 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.2/440.2 kB 9.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 22.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.7/367.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: openai
    Found existing installation: openai 1.70.0
    Uninstalling openai-1.70.0:
      Successfully uninstalled openai-1.70.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.23
    Uninstalling langsmith-0.3.23:
      Successfully uninstalled langsmith-0.3.23
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.50
    U

In [2]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re
import typing as tp
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
# from langchain_core.documents import Document
# from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.vectorstores import Qdrant as LCQdrant
from tqdm import tqdm


2025-07-02 03:15:51.590109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751426151.877094      93 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751426151.957691      93 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# COLLECTION_NAME = "freecad_docs"
# COLLECTION_NAME = "freecad_docs_hybrid"
# DATA_PATH = "data"
# DOCS_PATH = os.path.join(DATA_PATH, "docs")
# EMBEDDINGS_PATH = "embeddings"
# QDRANT_PATH = "qdrant"

COLLECTION_NAME = "freecad_docs_hybrid"
DATA_PATH = "/kaggle/input"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "embeddings"
QDRANT_PATH = "/kaggle/working/freecad-qdrant"

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

# Metrics

In [5]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [6]:
def mean_average_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(sum(sims) / len(sims) for sims in similarities) / len(similarities)

In [7]:
def mean_max_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(max(sims) for sims in similarities) / len(similarities)

In [8]:
def hit_rate_at_k(similarities: tp.List[tp.List[float]], threshold: float, k: int) -> float:
    hits = 0
    for sims in similarities:
        top_k = sims[:k]
        if any(sim >= threshold for sim in top_k):
            hits += 1
    return hits / len(similarities)

# Qdrant init

In [5]:
!cp -r /kaggle/input/freecad-qdrant/qdrant /kaggle/working/freecad-qdrant

In [10]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

In [11]:
client = QdrantClient(path=QDRANT_PATH)

In [12]:
def filter_code(doc: str) -> str:
    code_blocks = re.findall(r"```(?:[a-zA-Z]+\n)?(.*?)```", doc, re.DOTALL)
    code = "\n\n".join(code_blocks)
    return re.sub(r'\n{3,}', '\n\n', code)

In [ ]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench/tooltips_dataset.csv"))
# query_df = pd.read_csv(os.path.join(DATA_PATH, "tooltips_dataset.csv"))


# Qdrant LC init

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = LCQdrant.from_existing_collection(
    path="qdrant",
    embedding=embeddings,
    collection_name="freecad_docs_lc",
)

In [5]:
vectordb.similarity_search("Window")

[Document(metadata={'source': 'Reinforcement_ColumnRebars.wikitext', 'chunk_idx': 6, '_id': 'e6546e97-2147-4dab-a27d-1e162f8902ec', '_collection_name': 'freecad_docs_lc'}, page_content='Example'),
 Document(metadata={'source': 'Arch_Site.wikitext', 'chunk_idx': 2, '_id': '703319b2-08ac-44d3-aa6c-9b12a0b95317', '_collection_name': 'freecad_docs_lc'}, page_content='Properties\n\n Data'),
 Document(metadata={'source': 'Arch_Window.wikitext', 'chunk_idx': 10, '_id': 'a2369c9d-973b-44c0-9821-69aeff535f1c', '_collection_name': 'freecad_docs_lc'}, page_content='Our window type is now ready. We can create window objects from it, simply by selecting the App Part and pressing the window button. The "Height", "Width", "Subvolume" and "Tag" properties of the window will be linked to the corresponding property of the App Part, if existing.\n\nMaterials\n\nTo build a material for type-based windows:\n Create a multi-material\n Create one entry in the multi-material for each component of your App Par

# BM25 Baseline

In [ ]:
docs_df = pd.read_csv(os.path.join("chunks_df.csv"))
# docs_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df.csv"))

corpus = docs_df["content"].values

tokenized_corpus = []
for doc in corpus:
    # tokenized_corpus.append(word_tokenize(filter_code(str(doc))))
    tokenized_corpus.append(word_tokenize(str(doc)))


['▁Description', '▁It', '▁will', '▁serve', '▁to', '▁generate', '▁flat', ',', '▁shape', '-', 'based', '▁views', '▁from', '▁a', '▁Mes', 'h', '▁based', '▁object', ',', '▁to', '▁be', '▁used', '▁by', '▁the', '▁Arch', '▁Equip', 'ment', '▁tool', '.', '▁Usa', 'ge', '▁Select', '▁a', '▁Mes', 'h', '▁object', '.', '▁Select', '▁the', '▁button', ',', '▁or', '▁Arch', '▁→', '▁Utili', 'ties', '▁→', '▁3', 'View', 's', '▁from', '▁the', '▁top', '▁menu', '.', '▁Script', 'ing', '▁Arch', '▁API', '▁and', '▁Free', 'CAD', '▁Script', 'ing', '▁Basic', 's', '.', '▁This', '▁tool', '▁can', '▁be', '▁used', '▁in', '▁macro', 's', '▁and', '▁from', '▁the', '▁Python', '▁console', '▁by', '▁using', '▁the', '▁following', '▁function', ':', '▁``', '`', 'start', '_', 'code', '▁shape', '▁=', '▁create', 'M', 'esh', 'View', '(', 'ob', 'j', ',', '▁direction', '=', 'Free', 'CAD', '.', 'Ve', 'ctor', '(', '0', ',', '▁0', ',', '▁', '-1)', ',', '▁out', 'eron', 'ly', '=', 'Fa', 'lse', ',', '▁largest', 'on', 'ly', '=', 'Fa', 'lse', ')', '

In [ ]:
def bm25_retrieve(bm25, query: str, k: int = 5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [corpus[i] for i in top_k_idx], [scores[i] for i in top_k_idx]

In [ ]:
def bm25_retrieve_with_score(bm25, queries: np.ndarray, y: np.ndarray, k: int = 5):
    retrieved_texts = []
    scores = []

    for i in tqdm(range(len(queries))):
        results, _ = bm25_retrieve(bm25, queries[i], k=k)
        retrieved_texts.append(results)

        retrieved_vecs = embedding_model.encode(results)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(vec, y_vec) for vec in retrieved_vecs]
        scores.append(scores_batch)

    return retrieved_texts, scores

In [ ]:
def bm25_grid_search(tokenized_corpus, queries, y, k1_range, b_range, metric_fn):
    best_score = -float("inf")
    best_params = {}
    for k1 in tqdm(k1_range):
        for b in b_range:
            bm25 = BM25Okapi(tokenized_corpus, k1=k1, b=b)
            retrieved_texts = []
            scores = []
            for query in queries:
                tokenized_query = query.split()
                bm25_scores = bm25.get_scores(tokenized_query)
                top_k_idx = np.argsort(bm25_scores)[::-1][:5]
                retrieved_texts.append([corpus[i] for i in top_k_idx])
                scores.append([bm25_scores[i] for i in top_k_idx])
            score = metric_fn(scores)
            # print(f"k1={k1:.2f}, b={b:.2f}, score={score:.4f}")
            if score > best_score:
                best_score = score
                best_params = {'k1': k1, 'b': b}
    print("Best BM25 params:", best_params, "score:", best_score)
    return best_params, best_score

# Experiments

In [26]:
query_idx = 0

query = query_df.iloc[query_idx]["X"]
query_qdrant = query + " python code"
# query_qdrant = "Add window"
# query_qdrant = "Make a circle"


query_vec = embedding_model.encode(query_qdrant)
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)

for _ in range(len(hits.points)):
    print(hits.points[_].payload['source'])

retrieved_text_batch = [point.payload['content'] for point in hits.points]
retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]

retrieved_vec = embedding_model.encode(retrieved_code_batch)

y_vec = embedding_model.encode(query_df.iloc[query_idx]["Y"])

print(f"X: {query_df.iloc[query_idx]['X']}")
print("Y\n", query_df.iloc[query_idx]["Y"], "\n")
for i, vec in enumerate(retrieved_vec):
    print(f"Hit {i+1}, score: {cosine_similarity(vec, y_vec)}")

for i, text in enumerate(retrieved_text_batch):
    print(f"=================Hit {i+1}=================")
    print(text)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Arch_Wall.wikitext
Arch_CurtainWall.wikitext
Arch_3Views.wikitext
Arch_CurtainWall.wikitext
Arch_CutPlane.wikitext


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

X: Create a wall
Y
 import Arch
obj = Arch.makeWall() 

Hit 1, score: 0.6794919967651367
Hit 2, score: 0.6343599557876587
Hit 3, score: 0.674802303314209
Hit 4, score: 0.48535436391830444
Hit 5, score: 0.661526620388031
=================Hit 1=================
Scripting

 Arch API and FreeCAD Scripting Basics.

The Wall tool can be used in macros and from the Python console by using the following function:
```start_code
Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")
end_code```
Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.
 If no  is given, you can provide the numerical values for the ,  (thickness), and .
 If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.
  can be ,  or .
 It returns  if the operation fails.

Example:
=================Hit 2=================
Scripting

 Arch API and FreeCAD Scr

In [14]:
def retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int):
    for i in range(len(queries)):
        queries[i] += " python code"

    query_vectors = embedding_model.encode(queries)

    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vectors[i],
            limit=k,
        )

        retrieved_text_batch = [point.payload['content'] for point in hits.points]
        retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]


        retrieved_vec = embedding_model.encode(retrieved_code_batch)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(x_vec, y_vec) for x_vec in retrieved_vec]

        scores.append(scores_batch)
        retrieved_texts.append(retrieved_text_batch)

    return retrieved_texts, scores

In [15]:
queries = query_df["X"].values
y = query_df["Y"].values
texts, scores = retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

In [ ]:
# import json

# retrieved_df = query_df.copy()
# retrieved_df["retrieved"] = texts
# retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.dumps)
# retrieved_df.to_csv("bench_retrieved_hybrid.csv", index=False)

In [1]:
os

NameError: name 'os' is not defined

In [ ]:
queries = query_df["X"].values
y = query_df["Y"].values

best_bm25_params, best_bm25_score = bm25_grid_search(
    tokenized_corpus=tokenized_corpus,
    queries=queries,
    y=y,
    k1_range=np.arange(0.8, 2.2, 0.2),
    b_range = np.arange(0.3, 1.1, 0.1),
    metric_fn = mean_average_cosine_similarity
)

In [ ]:
queries = query_df["X"].values
y = query_df["Y"].values

best_bm25_params = {'k1': 1.9999999999999998, 'b': 0.3}

best_bm25 = BM25Okapi(tokenized_corpus, k1=best_bm25_params["k1"], b=best_bm25_params["b"])

bm_texts, bm_scores = bm25_retrieve_with_score(
    best_bm25,
    queries=queries,
    y=y,
    k=5
)

In [19]:
# Qdrant
# MACS 0.5307003046751022
# MMCS 0.6132178441286087
# Hit rate cosine 0.604

print("Qdrant")
print("MACS", mean_average_cosine_similarity(scores))
print("MMCS", mean_max_cosine_similarity(scores))
print("Hit rate cosine", hit_rate_at_k(scores, 0.6, 5))

Qdrant
MACS 0.5339390434503553
MMCS 0.6152037699222564
Hit rate cosine 0.608


In [128]:
print("BM25")
print("MACS", mean_average_cosine_similarity(bm_scores))
print("MMCS", mean_max_cosine_similarity(bm_scores))

BM25
MACS 0.46777109932899463
MMCS 0.5069063054323196


# Qwen LLM

In [17]:
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

qwen_model = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    qwen_model,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    qwen_model,
    trust_remote_code=True,
    device_map=device
)

text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024
)
llm = HuggingFacePipeline(pipeline=text_gen)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda


In [6]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
"""
)

def format_prompt(user_message: str, context: str = None) -> str:
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )

In [9]:
def get_context_from_hits(hits, max_chunks=3):
    """Extract and concatenate content from top hits."""
    docs = []
    for point in hits.points[:max_chunks]:
        docs.append(point.payload['content'])
    return "\n\n".join(docs)

def get_context_from_texts(texts):
    return "\n\n".join(texts)

def rag_query(query, client, embedding_model, collection_name, llm, top_k=3):
    query_vec = embedding_model.encode([query])[0]
    hits = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=top_k,
    )

    context = get_context_from_hits(hits, max_chunks=top_k)

    prompt = format_prompt(query, context)
    response = llm.invoke(prompt)
    return response

In [27]:
query = "Add window"
response = rag_query(query, client, embedding_model, "freecad_docs", llm)
print(response)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_76/1775823008.py:22: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(prompt)


<|im_start|>system

Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.

Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:

Explanation:
<|im_end|>
<|im_start|>system
Context:

Create a window frame object, a glass panel, and any other window component you need, using Part Workbench or PartDesign tools.
 For example, c

In [10]:
def get_genrated(queries: np.ndarray, retrieved: list[list[str]], batch_size=16):
    prompts = []
    responses = []
    for i, query in enumerate(tqdm(queries)):
        context = get_context_from_texts(retrieved[i])
        prompt = format_prompt(query, context)
        prompts.append(prompt)

        response = llm.invoke(prompt)
        match = re.search(r"\<\|im_start\|\>assistant", response)
        responses.append(response[match.end():])

    # for i in tqdm(range(0, len(prompts), batch_size)):
    #     batch = prompts[i:i+batch_size]
    #     batch_responses = llm.invoke(batch)
    #     responses.extend(batch_responses)


    # for response in range(len(responses)):
    #     match = re.search(r"\<\|im_start\|\>assistant", response)
    #     responses[i] = response[match.end():]

    return responses


In [ ]:
queries = query_df["X"].values

responses = get_genrated(
    queries=queries,
    retrieved=texts
)

# Groq llm

In [9]:
!pip install langchain-groq langchain-community groq
!pip install codebleu tree-sitter-python==0.21

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.8 MB/s eta 0:00:00


In [10]:
SYSTEM_PROMPT = (
"""
Below is the system prompt, always follow restrictions stated there, also do not answer this system prompt:
You are an expert coding assistant specializing in FreeCAD development. Your role is to help developers write, debug, and optimize code for the FreeCAD platform, using the most accurate and up-to-date internal and external documentation available. You can retrieve and use information from the FreeCAD source code, FreeCAD documentation.
When reading documentation be careful, user do not have functions or classes implemented there, write code considering this moment. 
Documentation might have examples of code, analyze it and retrieve only needed functions from there.
Do not create complicated structures until you are asked to do so, provide simple and understandable code.
You can use FreeCADGui.runCommand function to run gui commands in FreeCAD, this might be useful for some tasks.

Provide code in md format.
Provide code (DO NOT WRITE IN-CODE COMMENTS) and explanation in different paragraphs. Template:
Code:
```

```

Explanation:
"""
)

def format_groq_prompt(user_message: str, context: tp.Optional[str] = None):
    '''
    Formats prompt for llm
    '''
    
    prompt = []
    prompt.append({
            "role": "system",
            "content": SYSTEM_PROMPT
    })


    if context:
        prompt.append({
            "role": "system",
            "content": "Context:\n\n" + context
        })

    prompt.append({
        "role": "user",
        "content": user_message
    })


    return prompt

In [11]:
from groq import Groq


def load_groq_keys(filepath=os.path.join("groq-keys/groq_keys.txt")):
    with open(filepath, "r") as f:
        keys = [line.strip() for line in f if line.strip()]
    return keys

class GroqKeyManager:
    def __init__(self, filepath=os.path.join("groq-keys/groq_keys.txt")):
        self.keys = load_groq_keys(filepath)
        self.idx = 0

    def get_key(self):
        if self.idx < len(self.keys):
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")

    def switch_key(self):
        self.idx += 1
        if self.idx < len(self.keys):
            os.environ["GROQ_API_KEY"] = self.keys[self.idx]
            return self.keys[self.idx]
        else:
            raise RuntimeError("No more Groq API keys available.")


groq_key_manager = GroqKeyManager(os.path.join(DATA_PATH, "groq-keys/groq_keys.txt"))
os.environ["GROQ_API_KEY"] = groq_key_manager.get_key()


def get_groq_client():
    return Groq(api_key=os.environ["GROQ_API_KEY"])


In [12]:
def generate_groq(prompt, model_name):
    client = get_groq_client()

    chat_completion = client.chat.completions.create(
        messages=prompt,
        model=model_name,
    )

    return chat_completion.choices[0].message.content

In [15]:
from time import sleep

responses = []
last_key_idx = 0

def get_groq_genrated(
        model_name: str,
        queries: np.ndarray,
        retrieved: tp.Optional[list[list[str]]]
    ):
    prompts = []
    global client


    for i, query in enumerate(tqdm(queries)):
        if retrieved is not None:
            context = get_context_from_texts(retrieved[i])
        else:
            context = None
        prompt = format_groq_prompt(query, context)
        prompts.append(prompt)

        error_patience = 0
        while True:
            try:
                responses.append(generate_groq(prompt, model_name))
                break
            except Exception as e:
                if "rate limit" in str(e).lower() or "TPD" in str(e).lower():
                    try:
                        groq_key_manager.switch_key()
                        client = get_groq_client()
                        print(f"Switched to next GROQ API key: {groq_key_manager.idx}")
                    except RuntimeError:
                        print("All GROQ API keys exhausted.")
                        responses.append(None)
                        break
                else:
                    error_patience += 1
                    print(f"Error: {e}")
                    sleep(1)
                    if error_patience >= 10:
                        error_patience = 0
                        responses.append(None)
                        break


In [15]:
import json

retrieved_df = pd.read_csv(os.path.join(DATA_PATH, "bench-retrieved-hybrid/bench_retrieved_hybrid.csv"))
retrieved_df["retrieved"] = retrieved_df["retrieved"].apply(json.loads)

retrieved = retrieved_df["retrieved"].values
retrieved[0][0]

'Scripting\n\n Arch API and FreeCAD Scripting Basics.\n\nThe Wall tool can be used in macros and from the Python console by using the following function:\n\n```start_code\n\nWall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")\n\nend_code```\n\n Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.\n If no  is given, you can provide the numerical values for the ,  (thickness), and .\n If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.\n  can be ,  or .\n It returns  if the operation fails.\n\nExample:\n\n```start_code\n\nimport FreeCAD, Draft, Arch\n\np1 = FreeCAD.Vector(0, 0, 0)\np2 = FreeCAD.Vector(2000, 0, 0)\nbaseline = Draft.makeLine(p1, p2)\nWall1 = Arch.makeWall(baseline, length=None, width=150, height=2000)\nFreeCAD.ActiveDocument.recompute()end_code```'

In [16]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench-ds/tooltips_dataset.csv"))

queries = query_df["X"].values

get_groq_genrated(
    model_name="llama-3.3-70b-versatile",
    queries=queries,
    retrieved=retrieved
)

  8%|▊         | 21/250 [01:50<33:51,  8.87s/it]

Switched to next GROQ API key: 1


 16%|█▋        | 41/250 [03:28<31:00,  8.90s/it]

Switched to next GROQ API key: 2


 25%|██▍       | 62/250 [05:11<24:33,  7.84s/it]

...

 60%|█████▉    | 149/250 [14:34<13:49,  8.21s/it]

Switched to next GROQ API key: 5


 85%|████████▌ | 213/250 [21:57<04:12,  6.83s/it]

Switched to next GROQ API key: 6


100%|██████████| 250/250 [25:23<00:00,  6.09s/it]


In [17]:
with open("generted-70b-hybrid-split.json", "w") as f:
    json.dump(responses, f)

In [13]:
import json
with open("generted-70b-hybrid-split.json", "r") as f:
    generated = json.load(f)

# Evaluate generation

In [14]:
def extract_python_code(generated):
    code_blocks = []
    pattern = re.compile(r"```(?:python)?\n(.*?)```", re.DOTALL | re.IGNORECASE)
    for text in generated:
        if not isinstance(text, str):
            code_blocks.append(None)
            continue
        match = pattern.search(text)
        if match:
            code_blocks.append(match.group(1).strip())
        else:
            alt_pattern = re.compile(r"```start_code(.*?)end_code```", re.DOTALL | re.IGNORECASE)
            match = alt_pattern.search(text)
            if match:
                code_blocks.append(match.group(1).strip())
            else:
                code_blocks.append(None)
    return code_blocks

In [15]:
import ast

class VariableAndNumberNormalizer(ast.NodeTransformer):
    def __init__(self):
        self.var_map = {}
        self.imported_names = set()

    def visit_Import(self, node):
        for alias in node.names:
            self.imported_names.add(alias.asname or alias.name.split('.')[0])
        return node

    def visit_ImportFrom(self, node):
        for alias in node.names:
            self.imported_names.add(alias.asname or alias.name)
        return node

    def visit_Name(self, node):
        if node.id in self.imported_names:
            return node
        if isinstance(node.ctx, (ast.Store, ast.Load)):
            if node.id not in self.var_map:
                self.var_map[node.id] = f"var{len(self.var_map)+1}"
            node.id = self.var_map[node.id]
        return node

    def visit_arg(self, node):
        if node.arg not in self.var_map:
            self.var_map[node.arg] = f"var{len(self.var_map)+1}"
        node.arg = self.var_map[node.arg]
        return node

    def visit_Num(self, node):
        return ast.copy_location(ast.Constant(value=0), node)

    def visit_Constant(self, node):
        if isinstance(node.value, (int, float)):
            return ast.copy_location(ast.Constant(value=0), node)
        return node

def normalize_code(code_str):
    try:
        tree = ast.parse(code_str)
        normalizer = VariableAndNumberNormalizer()
        normalized_tree = normalizer.visit(tree)
        ast.fix_missing_locations(normalized_tree)
        return ast.unparse(normalized_tree)
    except Exception as e:
        print(f"Error normalizing code: {e}")
        print("================================\n")
        print(code_str)
        raise ValueError()
        return code_str

In [16]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "freecad-bench-ds/tooltips_dataset.csv"))

In [17]:
generated_code = extract_python_code(generated)

In [18]:
none_idx = [i for i in range(len(generated_code)) if generated_code[i] is None]
generated_code = [_ for _ in generated_code if _ is not None]
query_df = query_df.drop(none_idx).reset_index(drop=True)

In [19]:
ground_truth = query_df["Y"].values

In [410]:
# for i in range(len(ground_truth)):
#     if ground_truth[i] is not None:
#         ground_truth[i] = ground_truth[i].replace("‘", "'")
#         ground_truth[i] = ground_truth[i].replace("’", "'")

# query_df["Y"] = ground_truth
# query_df.to_csv(os.path.join("tooltips_dataset.csv"), index=False)

In [20]:
normalized_generated_code = []
normalized_bench_code = []

for i in range(len(generated_code)):
    try:
        normalized_generated_code.append(normalize_code(generated_code[i]))
    except:
        normalized_generated_code.append("")

    try:
        normalized_bench_code.append(normalize_code(ground_truth[i]))
    except:
        normalized_bench_code.append("")


Error normalizing code: invalid syntax (<unknown>, line 9)

import FreeCAD
from FreeCAD import Base

# Create a new document
doc = FreeCAD.newDocument()

# Create a column
column = doc.addObject("Part::Feature", "Column")
column_shape = column.Shape =.Base.Shape()

# Define the column's dimensions
height = 3000
width = 200
depth = 200

# Create the column's shape
column_shape = column.Shape = Part.makeBox(width, depth, height)

# Place the column at the origin
column.Placement.Base = FreeCAD.Vector(0, 0, 0)

# Add the column to the document
doc.addObject(column)

# Create a BIM object from the column
bim_column = doc.addObject("BIM::IFCObject", "BIM_Column")
bim_column.IfcType = "IfcColumn"
bim_column.Shape = column_shape
bim_column.PredefinedType = "COLUMN"

# Set the BIM column's properties
bim_column.addExtension("BIM_Properties", 
                        "IfcRelDefinesByProperties",
                        "Pset_ColumnCommon",
                        {"Name": "Column_1", 
         

In [21]:
from codebleu import calc_codebleu

result = calc_codebleu(
    ground_truth,
    generated_code,
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

{'codebleu': 0.2503453203713354, 'ngram_match_score': 0.00012643753128297063, 'weighted_ngram_match_score': 0.0031072976427163777, 'syntax_match_score': 0.39226519337016574, 'dataflow_match_score': 0.6058823529411764}


In [22]:
result = calc_codebleu(
    normalized_bench_code,
    normalized_generated_code,
    lang="python",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)
print(result)

{'codebleu': 0.2419047138962931, 'ngram_match_score': 0.002238780840962268, 'weighted_ngram_match_score': 0.03342254034923656, 'syntax_match_score': 0.3848987108655617, 'dataflow_match_score': 0.5470588235294118}


In [23]:
def codebleu_per_pair(refs, preds, weights=(0.25, 0.25, 0.25, 0.25), tokenizer=None):

    scores = []
    syntax_match_scores = []
    dataflow_match_scores = []
    for ref, pred in zip(refs, preds):
        try:
            code_bleu = calc_codebleu([ref], [pred], lang="python", weights=weights, tokenizer=tokenizer)
            score = code_bleu["codebleu"]
            syntax_match_score = code_bleu['syntax_match_score']
            dataflow_match_score = code_bleu['dataflow_match_score']
        except Exception as e:
            score = 0.0
            syntax_match_score = 0.0
            dataflow_match_score = 0.0
        scores.append(score)
        syntax_match_scores.append(syntax_match_score)
        dataflow_match_scores.append(dataflow_match_score)
    return np.array(scores), np.array(syntax_match_scores), np.array(dataflow_match_scores)

In [24]:
pair_bleu = codebleu_per_pair(
    ground_truth,
    generated_code
)

In [27]:
pair_bleu[1].mean()

0.3848388167388167

In [432]:
generated_clean = [_ for _ in generated if _ is not None]

In [28]:
def extract_calls_with_str_args(code):
    calls = set()
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                func = node.func
                names = []
                while isinstance(func, ast.Attribute):
                    names.append(func.attr)
                    func = func.value
                if isinstance(func, ast.Name):
                    names.append(func.id)
                    call_chain = ".".join(reversed(names))
                else:
                    continue
                str_args = set()
                for arg in node.args:
                    if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                        str_args.add(arg.value)
                calls.add((call_chain, frozenset(str_args)))
    except Exception:
        return None
    return calls

def is_relevant_call(answer_code, ground_truth_code):
    gt_calls = extract_calls_with_str_args(ground_truth_code)
    gen_calls = extract_calls_with_str_args(answer_code)

    if gen_calls is None:
        return 0

    for gt_chain, gt_str_args in gt_calls:
        for ans_chain, ans_str_args in gen_calls:
            if gt_chain == ans_chain:
                if gt_str_args:
                    if gt_str_args & ans_str_args:
                        return 1
                else:
                    return 1
    return 0

In [29]:
def get_relevance(ground_truth, generated_code):
    relevance = []
    for gen_code, gt_code in zip(generated_code, ground_truth):
        try:
            flag = is_relevant_call(gen_code, gt_code)
        except Exception:
            flag = 0
        relevance.append(flag)
    return np.array(relevance)

In [30]:
relevance = get_relevance(ground_truth, generated_code)

In [33]:
print(relevance.sum())

12
